In [1]:
import cudf
import numpy as np
import plotly.graph_objects as go
from tqdm import tqdm
import os
from scipy.stats import gaussian_kde
%load_ext cudf.pandas
import pandas as pd
import polars as pl
# Fonction pour charger les données parquet
def load_parquet(stock, date):
    file_path = f'/home/janis/3A/EA/HFT_QR_RL/data/smash4/DB_MBP_10/{stock}/{stock}_{date}.parquet'
    return cudf.read_parquet(file_path)

# Spécifier les dates et stocks
# Get all dates from the parquet files in the folder
folder_path = '/home/janis/3A/EA/HFT_QR_RL/data/smash4/DB_MBP_10/AAL'
dates = [f.split('_')[1].split('.')[0] for f in os.listdir(folder_path) if f.endswith('.parquet')]
dates.sort() # Sort dates chronologically
stocks = ["CXW"]


In [2]:
dates = dates[:5]

In [3]:
df = load_parquet('CXW', '2024-09-18')
df.head()

,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,flags,...,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol,date
ts_recv,,,,,,,,,,,,,,,,,,,,,
2024-09-18 08:05:35.963616571,2024-09-18 08:05:35.963450908,10,2,3981,A,N,0,5.35,100,130,...,0,0,<NA>,<NA>,0,0,0,0,CXW,2024-09-18
2024-09-18 08:05:35.963624101,2024-09-18 08:05:35.963458460,10,2,3981,A,A,0,21.37,100,130,...,0,0,<NA>,<NA>,0,0,0,0,CXW,2024-09-18
2024-09-18 08:05:35.964458631,2024-09-18 08:05:35.964293022,10,2,3981,A,B,0,5.35,100,130,...,0,0,<NA>,<NA>,0,0,0,0,CXW,2024-09-18
2024-09-18 08:05:35.964469423,2024-09-18 08:05:35.964303909,10,2,3981,A,A,0,21.37,100,130,...,0,0,<NA>,<NA>,0,0,0,0,CXW,2024-09-18
2024-09-18 08:09:59.697885830,2024-09-18 08:09:59.697720113,10,2,3981,C,B,0,5.35,100,130,...,0,0,<NA>,<NA>,0,0,0,0,CXW,2024-09-18


In [4]:
# Reload cudf.pandas extension
%reload_ext cudf.pandas

# Convert cudf DataFrame to pandas DataFrame first
df_pandas = df.to_pandas()

# Convert pandas DataFrame to polars DataFrame
df_polars = pl.DataFrame(df_pandas)


/home/janis/3A/EA/HFT_QR_RL/.venv/lib/python3.12/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=pool
  warnings.warn(


In [5]:
# Filter out unnecessary columns including bid/ask counts
columns_to_drop = ['flags', 'ts_in_delta', 'date', 'symbol', 'publisher_id', 'rtype', 'instrument_id'] + \
                 [f'bid_ct_{i:02d}' for i in range(10)] + \
                 [f'ask_ct_{i:02d}' for i in range(10)]
df_polars = df_polars.drop(columns_to_drop)

# Remove entries where side is "N"
df_polars = df_polars.filter(pl.col("side") != "N")

# Filter for events between 12h and 20h
df_polars = df_polars.filter(
    (pl.col("ts_event").dt.hour() >= 13) & 
    (pl.col("ts_event").dt.hour() < 20)
)

# Display the first few rows of filtered dataset
print("Shape after filtering:", df_polars.shape)
print("\nFirst few rows of filtered dataset:")
print(df_polars.head())


Shape after filtering: (37453, 48)

First few rows of filtered dataset:
shape: (5, 48)
┌─────────────────┬────────┬──────┬───────┬───┬───────────┬───────────┬───────────┬────────────────┐
│ ts_event        ┆ action ┆ side ┆ depth ┆ … ┆ ask_px_09 ┆ bid_sz_09 ┆ ask_sz_09 ┆ ts_recv        │
│ ---             ┆ ---    ┆ ---  ┆ ---   ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---            │
│ datetime[ns]    ┆ str    ┆ str  ┆ u8    ┆   ┆ f64       ┆ u32       ┆ u32       ┆ datetime[ns]   │
╞═════════════════╪════════╪══════╪═══════╪═══╪═══════════╪═══════════╪═══════════╪════════════════╡
│ 2024-09-18 13:0 ┆ A      ┆ B    ┆ 4     ┆ … ┆ null      ┆ 0         ┆ 0         ┆ 2024-09-18 13: │
│ 1:09.071461867  ┆        ┆      ┆       ┆   ┆           ┆           ┆           ┆ 01:09.07162766 │
│                 ┆        ┆      ┆       ┆   ┆           ┆           ┆           ┆ 1              │
│ 2024-09-18 13:0 ┆ A      ┆ A    ┆ 2     ┆ … ┆ null      ┆ 0         ┆ 0         ┆ 2024-09-18 13: │
│ 1:

In [6]:

# Initialize pygwalker with the polars DataFrame
import pygwalker as pyg
walker = pyg.walk(df_polars)


Box(children=(HTML(value='<div id="ifr-pyg-00062e6f871ac49fdB17EpGDX3sh5SbQ" style="height: auto">\n    <head>…

In [ ]:
# Calculate midprice
df_polars = df_polars.with_columns([
    ((pl.col("bid_px_00") + pl.col("ask_px_00")) / 2).alias("midprice")
])

# Convert to pandas for plotly
df_plot = df_polars.select(["ts_event", "bid_px_00", "ask_px_00", "midprice"]).to_pandas()

# Create the plot
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# Add traces
fig.add_trace(
    go.Scatter(x=df_plot["ts_event"], y=df_plot["bid_px_00"], 
               name="Best Bid", line=dict(color='green'))
)
fig.add_trace(
    go.Scatter(x=df_plot["ts_event"], y=df_plot["ask_px_00"], 
               name="Best Ask", line=dict(color='red'))
)
fig.add_trace(
    go.Scatter(x=df_plot["ts_event"], y=df_plot["midprice"], 
               name="Midprice", line=dict(color='black'))
)

# Update layout
fig.update_layout(
    title="Price Evolution Over Time",
    xaxis_title="Time",
    yaxis_title="Price",
    hovermode='x unified',
    showlegend=True
)

fig.show()
